# Generative Morphology (AE + Flow) on Euclid VIS 64px galaxies

End-to-end walkthrough of SHINE's learned-morphology tier on real Euclid VIS
data, from the raw quadrant image to a MAP fit of the observed galaxies:

1. **The data** — what a Euclid VIS quadrant frame actually contains
   (science / RMS / flag / background maps, PSF grid, MER catalogue).
2. **Source selection** — only the galaxies kept by the SHINE cuts *and*
   small enough to live on the 64x64 stamp tier (the tier the generative
   model was trained for); their cutouts are displayed.
3. **The generative model** — the frozen AutoEncoder + normalizing flow
   (`shine.morphology`), loaded from checkpoints.
4. **10 galaxies without PSF** — `z ~ flow`, `g = AE.decode(z)`, plus the
   checks that the prior actually sampled is the one the flow learned.
5. **The same 10 galaxies with residual PSFs** — the residual-PSF grid
   `PSF_3-4-F_residual.fits.gz` bilinearly interpolated at 10 detector
   positions (the PSFs used are displayed too).
6. **Rendering conventions** — why the fitted galaxies came out rotated with
   high-frequency structure: which coordinate frame `decode(z)` lives in, and
   three tests that isolate the frame, the pixel response and the FFT grid.
7. **Four models, same galaxies** — per-galaxy MAP with the shear held at
   zero, comparing the library renderer against the training-frame one, with
   and without a free flux parameter. This is the experiment that says what
   fixes the fit.
8. **A shear fit under the repaired model** — one global `(g1, g2)` shared by
   every galaxy and exposure, on stamps.
9. **The library path as it stands** — `MultiExposureScene` +
   `Inference.run_map` on the full frames, for comparison.
10. **Findings and proposed library changes** — the summary.

> **Why the *residual* PSF and not the full local PSF?** The AE was trained
> with `decode(z)` convolved by `psf_residual`, i.e. the kernel relating the
> true local PSF to a fixed isotropic reference PSF (`PSFiso`). So
> `decode(z) ~= G_true (*) PSFiso`: the reference PSF is still baked into the
> decoder output. Convolving it with the *full* local PSF would apply
> `PSFiso` twice and over-blur the stamp. See
> `shine/morphology/psf_residual.py` and `data/LEARNED_MORPHOLOGY_NOTES.md`.

## 0. Setup (Colab, GPU T4)

This notebook is meant to run on Colab: `flowjax` + `JAX-GalSim` + the
pinned `equinox` are awkward to install alongside a local JAX. Run the two
cells below on a fresh runtime, then restart the runtime if Colab asks.

**Pick a GPU runtime** (*Runtime → Change runtime type → T4 GPU*) before
running anything. Everything here works on CPU, but the two MAP fits
(sections 6 and 7) re-render every galaxy through the AE decoder plus
JAX-GalSim FFTs at every optimisation step — on 64x64 stamps in section 6,
on all three 2048x2066 exposures in section 7 — which is what makes CPU
painful; a T4 has ample memory for these settings (the whole
scene is ~50 MB of float32 images). The setup cell below deliberately does
**not** pin `jax`, so Colab's CUDA-enabled build is kept — check that
`jax.devices()` reports a `cuda` device in the next cell; if it prints CPU
after the installs, `jax` was upgraded past its CUDA plugin, so restart the
runtime (or `pip install -q "jax[cuda12]"`) before continuing.

The repo carries both the Euclid test data (`data/EUC_VIS_SWL/`) and the
AE/flow checkpoints (`wandb_weights/`) through **git-lfs**, so `git lfs pull`
is required — without it those files are 130-byte pointer stubs and every
`fits.open` / checkpoint load below fails.

In [ ]:
# Dependencies. Every version here is pinned on purpose; letting pip resolve
# the whole stack freely can hit "resolution-too-deep" and never finish, and
# two of the pins are load-bearing:
#   * equinox 0.13.6 -- the version the checkpoints were serialised with.
#   * flowjax >= 18 -- the MAF/RQS flow checkpoint (9i28jqsm) was trained with
#     it. flowjax 18.0.0 changed RationalQuadraticSpline's parameterisation
#     (40 parameters per latent dimension at knots=12, against 38 in 17.x), so
#     loading that checkpoint under flowjax 17.x fails outright with a shape
#     mismatch: "leaf ... has changed shape from (8, 608, 128) to (8, 640, 128)".
# NB: do NOT pin paramax==0.0.4 -- flowjax requires paramax>=0.0.5 and the pin
# makes the resolver fail outright.
!pip install -q "equinox==0.13.6" "einops>=0.8,<0.9"
!pip install -q "flowjax==18.0.0" "paramax>=0.0.5"
# Pinned rather than installed from git HEAD: an unpinned HEAD is what makes
# this notebook silently stop reproducing. If drawImage raises
# GalSimIncompatibleValuesError ("shape of array is inconsistent with provided
# bounds"), the jax / JAX-GalSim pair is mismatched -- try another release here.
!pip install -q "jax-galsim==2026.2.0"


In [ ]:
# Clone SHINE with its LFS payload (data + AE/flow checkpoints), then install.
!git lfs install
!git clone -b GenGal64 https://github.com/VincentB03/SHINE.git SHINE
%cd SHINE
!git lfs pull
!pip install -q -e .

In [ ]:
import logging
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*complex128.*", module="jax_galsim")

import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

from shine.config import InferenceConfig, MAPConfig
from shine.euclid.config import (
    EuclidDataConfig,
    EuclidInferenceConfig,
    SourceSelectionConfig,
)
from shine.euclid.data_loader import EuclidDataLoader, EuclidPSFModel
from shine.euclid.plots import plot_exposure_comparison
from shine.euclid.scene import MultiExposureScene, render_model_images
from shine.inference import Inference
from shine.morphology.config import LearnedMorphologyConfig
from shine.morphology.loader import load_frozen_autoencoder, load_frozen_flow
from shine.morphology.render import render_learned_galaxy
import jax_galsim as galsim

%matplotlib inline

# Run from either the repo root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "shine").is_dir() else Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "EUC_VIS_SWL"
QUADRANT = "3-4.F"

# Frozen AE + flow checkpoints (committed in the repo, git-lfs).
AE_CHECKPOINT_DIR = str(REPO_ROOT / "wandb_weights" / "i344nq38" / "epoch_2000")
AE_EPOCH = 2000
FLOW_CHECKPOINT_DIR = str(REPO_ROOT / "wandb_weights" / "9i28jqsm" / "epoch_500")
FLOW_EPOCH = 500

# Source selection. This is a *faint band*, not the historical "everything
# above an SNR floor, then keep the brightest N" -- see section 2 for why.
# In short: the flow prior's decoded stamps sum to ~1.3e3 ADU, the core of the
# catalogue sits at ~1.1e3 ADU, and the top-20-by-SNR sources the notebook
# used to fit sit at ~7e4 ADU, i.e. a factor ~60 outside the decoder's trained
# domain. MAX_SOURCES is larger than before because the band is drawn at
# random from ~860 survivors and only ~40% of them land inside exposure 0.
MIN_SNR = 12.0
MAX_SNR = 25.0
MAX_SOURCES = 60
SELECTION_ORDER = "random"   # unbiased inside the band; "brightest" re-biases it
SELECTION_SEED = 0
MAP_STEPS = 150
LEARNING_RATE = 0.002
RNG_SEED = 42
N_SHOW = 10  # galaxies generated / PSFs displayed / stamps inspected

# The data loader logs the full source-selection cascade at INFO level.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
logging.getLogger("jax").setLevel(logging.WARNING)

print("Repo root:", REPO_ROOT)
print("Data dir :", DATA_DIR, "(exists:", DATA_DIR.is_dir(), ")")
print("JAX devices:", jax.devices())
if jax.devices()[0].platform != "gpu":
    print("WARNING: running on CPU -- the MAP fits (sections 6-7) will be slow. "
          "Runtime > Change runtime type > T4 GPU.")

# Guard against un-pulled git-lfs pointer stubs (a few hundred bytes instead
# of tens of MB) -- they produce confusing FITS/deserialisation errors later.
for path in [DATA_DIR / "PSF_3-4-F_residual.fits.gz",
             Path(AE_CHECKPOINT_DIR) / f"model_checkpoint_{AE_EPOCH}.eqx"]:
    assert path.exists(), f"missing: {path}"
    assert path.stat().st_size > 10_000, f"{path} looks like a git-lfs pointer -- run `git lfs pull`"

## 1. The data: a Euclid VIS quadrant frame

`data/EUC_VIS_SWL/` holds **real Euclid Q1 VIS data**: observation 2704
(NGC 6505, DEEP mode, 2024-07-18), quadrant **3-4.F** of the VIS focal
plane — the quadrant closest to the pointing centre — extracted from three
overlapping dithered exposures (560.52 s each) via the ESA Euclid Science
Archive.

Per exposure the file carries three 2048x2066 planes:

| Extension | Content |
|-----------|---------|
| `3-4.F.SCI` | Calibrated science image, **in ADU, not background-subtracted** |
| `3-4.F.RMS` | Per-pixel noise sigma (ADU) — the likelihood's weights |
| `3-4.F.FLG` | Data-quality bitmask (bad pixels, cosmic rays, ghosts, star halos) |

plus, as separate products:

- `EUC_VIS_SWL-BKG-*` — pipeline background maps (one per dither);
- `PSF_3-4-F.fits.gz` — the local PSF model as a 9x9 grid of 21x21 stamps
  tiled into one image, bilinearly interpolated at each source position;
- `PSF_3-4-F_residual.fits.gz` — the same grid divided (in Fourier space) by
  the isotropic reference PSF, i.e. the kernel the learned tier needs;
- `catalogue_3-4-F.fits.gz` — the MER catalogue (positions, fluxes, sizes,
  quality flags) driving source selection.

Pixel scale: 0.1"/px. Photometry: `m_AB = -2.5 log10(ADU) + MAGZEROP + 2.5 log10(EXPTIME)`.

**Background subtraction.** The SCI planes are *not* background-subtracted on
disk; SHINE does it at load time in
`EuclidExposure.prepare_image_data`, which is what fills
`ExposureSet.images`:

```python
image = self.sci - background_map      # when background_paths is configured
image = self.sci - sigma_clipped_median(self.sci[mask])   # fallback otherwise
```

So `data.images[j]` below is `SCI - BKG` **pixel by pixel** (not a scalar
level), and the scene model fits that, with no background term of its own —
hence the `background_paths=` argument in the next section: without it the
loader falls back to one sigma-clipped median for the whole quadrant and the
leftover straylight structure gets absorbed into the galaxy models. (The
`EuclidInferenceConfig.background` field — `"fit"`/`"median"`/`"fixed"` — is
declared but not read by any code path today; only `background_paths`
matters.) The same masking step turns `FLG & bad_pixel_mask` pixels into
`sigma = 1e10`, i.e. zero weight in the likelihood.

In [ ]:
exposure_paths = sorted(str(p) for p in DATA_DIR.glob("EUC_VIS_SWL-DET-*_3-4-F.fits.gz"))
bkg_paths = sorted(str(p) for p in DATA_DIR.glob("EUC_VIS_SWL-BKG-*_3-4-F.fits.gz"))
assert len(exposure_paths) == 3 and len(bkg_paths) == 3

# Inspect one raw exposure directly (before any SHINE processing).
with fits.open(exposure_paths[0]) as hdul:
    hdul.info()
    sci_hdu = hdul[f"{QUADRANT}.SCI"]
    sci_raw = sci_hdu.data.astype(np.float32)
    rms_raw = hdul[f"{QUADRANT}.RMS"].data.astype(np.float32)
    flg_raw = hdul[f"{QUADRANT}.FLG"].data.astype(np.int32)
    hdr = sci_hdu.header

with fits.open(bkg_paths[0]) as hdul:
    bkg_raw = hdul[QUADRANT].data.astype(np.float32)

print()
for key in ("EXPTIME", "GAIN", "RDNOISE", "MAGZEROP", "CTYPE1", "CTYPE2"):
    print(f"  {key:9s} = {hdr.get(key)}")
print(f"  shape     = {sci_raw.shape}")
print(f"  SCI range = [{sci_raw.min():.1f}, {sci_raw.max():.1f}] ADU")
print(f"  flagged   = {(flg_raw != 0).mean() * 100:.2f}% of pixels")

In [ ]:
# Displayed with an arcsinh stretch: a linear scale is dominated by the few
# bright stars and shows nothing of the faint galaxies we actually model.
# Second panel = exactly what prepare_image_data will hand to the model:
# the science plane minus the per-pixel background map.
panels = [
    (np.arcsinh(sci_raw), "SCI, raw [arcsinh(ADU)]", "gray_r"),
    (np.arcsinh(sci_raw - bkg_raw), "SCI - BKG map, per pixel [arcsinh(ADU)]", "gray_r"),
    (bkg_raw, "Background map [ADU]", "viridis"),
    (rms_raw, "RMS noise map [ADU]", "magma"),
    (flg_raw != 0, "Flagged pixels (FLG != 0)", "gray"),
]

fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 6))
for ax, (img, title, cmap) in zip(axes, panels):
    arr = np.asarray(img, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    vmin, vmax = np.percentile(finite, [1, 99]) if finite.size else (0, 1)
    im = ax.imshow(arr, origin="lower", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("x [px]")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel("y [px]")
fig.suptitle(
    f"Euclid VIS quadrant {QUADRANT} — obs 2704 (NGC 6505), dither 0 — "
    f"{sci_raw.shape[1]}x{sci_raw.shape[0]} px @ 0.1\"/px",
    fontsize=13,
)
fig.tight_layout()
plt.show()

## 2. Source selection: a faint band on the 64x64 tier

`EuclidDataLoader` applies the `SourceSelectionConfig` cuts to the MER
catalogue (SNR, VIS detection, spurious / point-source flags, detection
quality bitmask) and then assigns each surviving source a stamp tier from
`galaxy_stamp_sizes`: the smallest stamp that fits `~3 * hlr` on each side
plus the PSF half-width, i.e. `2 * (3 * hlr_px + 10.5) <= stamp`.

Here `galaxy_stamp_sizes = [64]`, which is exactly the "64x64 cut": sources
too extended for a 64px stamp are dropped by `_select_sources`, so **every**
remaining galaxy sits on the learned tier the AE was trained for
(64x64 px @ 0.1"/px).

**Which galaxies, though.** This notebook used to fit the `MAX_SOURCES`
*brightest* sources of the quadrant, because `_select_sources` sorted by SNR
descending and truncated. That is the worst possible choice for this model:

| selection | N with a full cutout in exp 0 | catalogue flux [ADU] |
|---|---|---|
| `min_snr=20`, top 20 by SNR (before) | 5 | median 7.4e4, range 4.0e4-1.3e5 |
| SNR band 12-25, 60 drawn at random (now) | 13 | median 8.7e2, range 4.2e2-2.5e3 |
| flow prior, `decode(z)` summed (section 4) | — | median 1.3e3, 5-95th 5.3e2-2.5e4 |

The third row is the point: the AE/flow pair was trained on a
magnitude-limited population, and the **core of the catalogue lands on top of
the prior**, while the bright tail sits ~60x outside it. The "400-1200x
flux gap" recorded in `data/LEARNED_MORPHOLOGY_NOTES.md` was in large part
this selection effect, not a unit mismatch.

Two new `SourceSelectionConfig` fields express that: `max_snr` closes the
band from above, and `selection_order="random"` makes `max_sources` an
unbiased draw from the survivors instead of a bright-end truncation (with
`selection_seed` for reproducibility).


In [ ]:
learned_morphology = LearnedMorphologyConfig(
    enabled=True,
    ae_checkpoint_dir=AE_CHECKPOINT_DIR,
    ae_epoch=AE_EPOCH,
    flow_checkpoint_dir=FLOW_CHECKPOINT_DIR,
    flow_epoch=FLOW_EPOCH,
    apply_to_stamp_size=64,
    # NOT PSF_3-4-F.fits.gz -- see the note at the top of the notebook.
    psf_residual_path=str(DATA_DIR / "PSF_3-4-F_residual.fits.gz"),
)

config = EuclidInferenceConfig(
    data=EuclidDataConfig(
        exposure_paths=exposure_paths,
        psf_path=str(DATA_DIR / "PSF_3-4-F.fits.gz"),
        catalog_path=str(DATA_DIR / "catalogue_3-4-F.fits.gz"),
        background_paths=bkg_paths,   # real per-pixel background, not a median
        quadrant=QUADRANT,
    ),
    sources=SourceSelectionConfig(
        min_snr=MIN_SNR,
        max_snr=MAX_SNR,                  # closes the band from above
        max_sources=MAX_SOURCES,
        selection_order=SELECTION_ORDER,  # unbiased draw, not a bright-end cut
        selection_seed=SELECTION_SEED,
    ),
    inference=InferenceConfig(
        method="map",
        map_config=MAPConfig(enabled=True, num_steps=MAP_STEPS, learning_rate=LEARNING_RATE),
        rng_seed=RNG_SEED,
    ),
    galaxy_stamp_sizes=[64],          # the 64x64 cut
    learned_morphology=learned_morphology,
)

data = EuclidDataLoader(config).load()

print()
print(f"Sources kept        : {data.n_sources} (all on the 64px learned tier)")
print(f"Exposures           : {data.n_exposures}  |  image {data.image_ny}x{data.image_nx}")
print(f"Catalog flux [ADU]  : [{float(data.catalog_flux_adu.min()):.0f}, {float(data.catalog_flux_adu.max()):.0f}]")
print(f"Catalog HLR [arcsec]: [{float(data.catalog_hlr_arcsec.min()):.3f}, {float(data.catalog_hlr_arcsec.max()):.3f}]")
print(f"Residual PSF stamps : {None if data.psf_residual_images is None else tuple(data.psf_residual_images.shape)}")
assert int(np.asarray(data.source_stamp_tier).max()) == 0, "some source is not on the 64px tier"

# The images the model is fitted to are the per-pixel background-subtracted
# science planes (see the note in section 1), not the raw SCI planes.
np.testing.assert_allclose(np.asarray(data.images[0]), sci_raw - bkg_raw, rtol=0, atol=1e-3)
print(f"\ndata.images[0] == SCI - BKG (per pixel) ✓   "
      f"median = {float(np.median(data.images[0])):.3f} ADU (raw SCI: {np.median(sci_raw):.1f} ADU)")

In [ ]:
pos0 = np.asarray(data.pixel_positions[:, 0, :])   # positions in exposure 0
visible0 = np.asarray(data.source_visible[:, 0])
image0 = np.asarray(data.images[0])                # background-subtracted

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))

axes[0].imshow(np.arcsinh(image0), origin="lower", cmap="gray_r",
               vmin=np.percentile(np.arcsinh(image0), 1),
               vmax=np.percentile(np.arcsinh(image0), 99.5), interpolation="nearest")
axes[0].scatter(pos0[visible0, 0], pos0[visible0, 1], s=60, facecolors="none",
                edgecolors="#F44336", linewidths=1.2, label="visible in exp 0")
axes[0].scatter(pos0[~visible0, 0], pos0[~visible0, 1], s=60, marker="x",
                color="#2196F3", label="outside exp 0")
axes[0].set_xlim(0, data.image_nx)
axes[0].set_ylim(0, data.image_ny)
axes[0].set_title(f"Selected galaxies (N={data.n_sources})")
axes[0].set_xlabel("x [px]"); axes[0].set_ylabel("y [px]")
axes[0].legend(fontsize=8, loc="upper right")

axes[1].hist(np.asarray(data.catalog_hlr_arcsec), bins=15, color="#4CAF50", edgecolor="k")
axes[1].set_xlabel("catalog HLR [arcsec]"); axes[1].set_ylabel("count")
axes[1].set_title("Size distribution (all fit in 64 px)")

axes[2].hist(np.log10(np.asarray(data.catalog_flux_adu)), bins=15, color="#FF9800", edgecolor="k")
axes[2].set_xlabel("log10(catalog flux [ADU])"); axes[2].set_ylabel("count")
axes[2].set_title("Flux distribution")

fig.tight_layout()
plt.show()

In [ ]:
STAMP = 64
half = STAMP // 2


def cutout(image, x, y, size=STAMP):
    # Square cutout centred on a (rounded) pixel position; None at the edge.
    xi, yi = int(round(float(x))), int(round(float(y)))
    y0, y1, x0, x1 = yi - size // 2, yi + size // 2, xi - size // 2, xi + size // 2
    if y0 < 0 or x0 < 0 or y1 > image.shape[0] or x1 > image.shape[1]:
        return None
    return image[y0:y1, x0:x1]


# Galaxies with a complete, non-truncated 64x64 cutout in exposure 0.
gal_indices = [
    i for i in range(data.n_sources)
    if visible0[i] and cutout(image0, *pos0[i]) is not None
]
print(f"{len(gal_indices)} / {data.n_sources} galaxies have a full 64x64 cutout in exposure 0")
assert gal_indices, "no galaxy with a complete cutout -- widen the SNR band or raise MAX_SOURCES"

# Sections 5 and 7 inspect these galaxies one by one; never ask for more
# than we actually have.
N_SHOW = min(N_SHOW, len(gal_indices))

n_cols = 5
n_rows = max(1, int(np.ceil(len(gal_indices) / n_cols)))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.4 * n_cols, 2.6 * n_rows))
for ax, i in zip(np.atleast_1d(axes).ravel(), gal_indices):
    stamp = cutout(image0, *pos0[i])
    ax.imshow(np.arcsinh(stamp), origin="lower", cmap="gray_r")
    ax.set_title(f"#{i} | {float(data.catalog_hlr_arcsec[i]):.2f}\"", fontsize=8)
    ax.axis("off")
for ax in np.atleast_1d(axes).ravel()[len(gal_indices):]:
    ax.axis("off")
fig.suptitle("Observed 64x64 cutouts of the selected galaxies (exposure 0, arcsinh)", fontsize=12)
fig.tight_layout()
plt.show()

## 3. The generative model (frozen AutoEncoder + normalizing flow)

`shine.morphology` holds two frozen `equinox` checkpoints:

- the **AutoEncoder**, whose decoder maps a latent code `z` to a 64x64
  galaxy light profile at 0.1"/px (softplus output, so non-negative);
- the **normalizing flow**, a learned prior over those latent codes — this
  is what makes the morphology a *probabilistic* model rather than a fixed
  template, and what `sample_latent_codes` turns into a NumPyro sample site.

Both are loaded in inference mode (dropout disabled), so `decode` is
deterministic, and neither is ever optimised during inference.

In [ ]:
ae = load_frozen_autoencoder(AE_CHECKPOINT_DIR, AE_EPOCH)
flow = load_frozen_flow(FLOW_CHECKPOINT_DIR, FLOW_EPOCH)

latent_flat = int(np.prod(flow.latent_dim))

print(f"AE   : stamp {ae.nx}x{ae.ny} px @ {ae.scale}\"/px")
print(f"Flow : latent_dim={list(flow.latent_dim)} ({latent_flat} values), cond_dim={flow.cond_dim}")

assert ae.nx == ae.ny == 64, "unexpected AE stamp size"
assert abs(ae.scale - config.data.pixel_scale) < 1e-6, "AE scale != data pixel scale"
assert flow.cond_dim is None, "sample_latent_codes only supports unconditional flows"
assert tuple(ae.encode(jnp.zeros((1, ae.nx, ae.ny)), key=None).shape) == tuple(flow.latent_dim), (
    "AE latent shape != flow.latent_dim -- this AE/flow pair was not trained together"
)

## 4. Generating 10 galaxies — no PSF

`z ~ flow`, `g = AE.decode(z)`. These are the intrinsic profiles the model
proposes: no instrument response applied yet (strictly: they still carry the
fixed isotropic reference PSF from training, which is exactly why section 5
convolves with the *residual* PSF and not the full one).

Two things to keep in mind while looking at them, because both make the
samples *look* wrong when they are not:

- **A Euclid galaxy is small.** At 0.1"/px a typical half-light radius of
  0.2-0.4" is 2-4 px, so the galaxy covers a few percent of a 64x64 stamp.
  On a linear scale every sample then looks like the same tiny dot; the
  mosaic below uses an arcsinh stretch on the full stamp.
- **The prior is a magnitude-limited population.** The flow learned the
  distribution of `VincentB03/euclid-Q1-VF`, dominated by faint, small,
  barely-resolved galaxies — so most draws *should* be small and similar.
  Section 2 now selects observed galaxies from the same part of the
  population (SNR band 12-25, drawn at random), so the flux histogram below
  should **overlap** rather than sit two decades apart. It did sit two
  decades apart while this notebook fitted the brightest sources of the
  quadrant; that comparison was never fair.

The cells also check the two generative paths agree — `flow.sample()` versus
what the inference actually uses, `sample_latent_codes` (`z_base` pushed
through `flow.forward`). They must produce the same distribution; they did
**not** before `shine.morphology.prior` was fixed to use the flow's own
(trained, non-standard) base `loc`/`scale`.

> **The flow is now MAF + rational-quadratic splines** (`9i28jqsm`), the
> architecture `train_flow.py` always specified, replacing the RealNVP with an
> affine transformer (`2815kuay`) shipped before it. Two things visibly
> improve, both re-measured by the cells below: this checkpoint's base stayed
> essentially standard (max `|loc|` 0.02, scales 0.90-1.04, against a `loc`
> component of −1.65 for the RealNVP), and **no** draw falls outside the
> `(−5, 5)` interval `softclip2` guarantees the decoder ever saw — a bounded
> spline transformer cannot leave it, whereas the affine flow put 0.01% of its
> own draws outside. The base fix in `shine/morphology/prior.py` stays
> necessary anyway: flowjax keeps `loc`/`scale` trainable, so the next
> checkpoint may drift again.

In [ ]:
# --- Diagnostic 1: do the two generative paths agree? ---------------------
# flow.sample() is the flow's own generative process. sample_latent_codes()
# (used by MultiExposureScene) instead draws z_base and pushes it through
# flow.forward, so it must apply the flow's *trained* base loc/scale --
# flowjax keeps those trainable, and this checkpoint's are not (0, 1).
import paramax

base = paramax.unwrap(flow.flow).base_dist
base_loc, base_scale = jnp.asarray(base.loc), jnp.asarray(base.scale)
print("flow base loc  :", np.round(np.asarray(base_loc), 3))
print("flow base scale:", np.round(np.asarray(base_scale), 3))
if np.abs(np.asarray(base_loc)).max() > 0.05:
    print("  -> NOT a standard normal: pushing N(0,1) through flow.forward "
          "would sample a different prior than the trained one.")

n_check = 4000
ref = np.asarray(flow.sample(key=jax.random.key(11), sample_shape=(n_check,)))
eps = jax.random.normal(jax.random.key(12), (n_check, latent_flat))
via_prior = np.asarray(jax.vmap(flow.forward)(base_loc + base_scale * eps))
naive = np.asarray(jax.vmap(flow.forward)(eps))          # the un-fixed path

for label, z in (("sample_latent_codes path", via_prior), ("naive N(0,1) path", naive)):
    print(f"{label:26s}: max |mean - flow.sample mean| = "
          f"{np.abs(z.mean(0) - ref.mean(0)).max():.3f}, "
          f"|z|>5 (outside the AE's softclip2 range): {(np.abs(z) > 5).mean() * 100:.2f}%")

In [ ]:
def second_moments(img, pixel_scale=0.1):
    # Unweighted second moments -> size sigma [arcsec], |e|, and the position
    # angle of the major axis [deg]. The angle is what section 6 uses to
    # measure how much a rendering convention rotates a galaxy.
    img = np.clip(np.asarray(img, dtype=np.float64), 0, None)
    total = img.sum()
    if total <= 0:
        return np.nan, np.nan, np.nan
    ny, nx = img.shape
    y, x = np.mgrid[0:ny, 0:nx]
    xc, yc = (x * img).sum() / total, (y * img).sum() / total
    dx, dy = x - xc, y - yc
    qxx = (img * dx ** 2).sum() / total
    qyy = (img * dy ** 2).sum() / total
    qxy = (img * dx * dy).sum() / total
    sigma = (max(qxx * qyy - qxy ** 2, 0.0)) ** 0.25 * pixel_scale
    denom = qxx + qyy
    if denom <= 0:
        return sigma, np.nan, np.nan
    e1, e2 = (qxx - qyy) / denom, 2 * qxy / denom
    return sigma, float(np.hypot(e1, e2)), float(np.degrees(0.5 * np.arctan2(e2, e1)))


z_gen = flow.unflatten_latent(flow.sample(key=jax.random.key(0), sample_shape=(N_SHOW,)))
galaxies_nopsf = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_gen)[:, 0]

print("z shape:", z_gen.shape, "| decoded shape:", galaxies_nopsf.shape)
assert jnp.all(jnp.isfinite(galaxies_nopsf)) and jnp.all(galaxies_nopsf >= 0)

fig, axes = plt.subplots(2, 5, figsize=(15, 6.8))
for k, ax in enumerate(axes.ravel()):
    img = np.asarray(galaxies_nopsf[k])
    sigma, e, angle = second_moments(img, config.data.pixel_scale)
    ax.imshow(np.arcsinh(img / max(img.max() * 1e-3, 1e-8)), origin="lower", cmap="inferno")
    ax.set_title(f"z#{k} | flux={img.sum():.0f}\nsigma={sigma:.2f}\" |e|={e:.2f} PA={angle:+.0f}deg",
                 fontsize=8)
    ax.axis("off")
fig.suptitle("10 generated galaxies — AE.decode(z), z ~ flow prior, no PSF (arcsinh stretch)",
             fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
# Is the sample diversity real, or does the prior collapse to one galaxy?
# Measured on a large batch, and compared against the observed selection.
z_many = flow.unflatten_latent(flow.sample(key=jax.random.key(5), sample_shape=(300,)))
gen_many = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_many)[:, 0]

gen_flux = np.asarray(gen_many.sum(axis=(1, 2)))
moments = np.array([second_moments(g, config.data.pixel_scale) for g in np.asarray(gen_many)])
gen_sigma, gen_e = moments[:, 0], moments[:, 1]

print(f"generated flux  [AE units]: median {np.median(gen_flux):.0f}, "
      f"5-95th [{np.percentile(gen_flux, 5):.0f}, {np.percentile(gen_flux, 95):.0f}]")
print(f"observed  flux  [ADU]     : median {float(np.median(data.catalog_flux_adu)):.0f}, "
      f"5-95th [{float(np.percentile(data.catalog_flux_adu, 5)):.0f}, "
      f"{float(np.percentile(data.catalog_flux_adu, 95)):.0f}]")
print(f"  -> ratio of medians: {float(np.median(data.catalog_flux_adu)) / np.median(gen_flux):.2f}x "
      "(near 1 means AE units ARE ADU and the old 400-1200x gap was selection)")
print(f"generated size  sigma [\"] : median {np.nanmedian(gen_sigma):.3f}, "
      f"5-95th [{np.nanpercentile(gen_sigma, 5):.3f}, {np.nanpercentile(gen_sigma, 95):.3f}]")
print(f"pixel-wise std across samples / mean peak: "
      f"{float(jnp.mean(jnp.std(gen_many, axis=0)) / jnp.mean(jnp.max(gen_many, axis=(1, 2)))):.4f} "
      "(a collapsed prior would sit near 0)")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].hist(np.log10(np.clip(gen_flux, 1e-3, None)), bins=25, color="#9C27B0",
             edgecolor="k", label="generated (flow prior)")
axes[0].hist(np.log10(np.asarray(data.catalog_flux_adu)), bins=15, color="#FF9800",
             edgecolor="k", alpha=0.6,
             label=f"selected sources ({data.n_sources}, SNR band {MIN_SNR:.0f}-{MAX_SNR:.0f})")
axes[0].set_xlabel("log10(total flux)"); axes[0].set_ylabel("count")
axes[0].set_title("Flux: prior population vs. the observed band"); axes[0].legend(fontsize=8)

axes[1].hist(gen_sigma[np.isfinite(gen_sigma)], bins=25, color="#4CAF50", edgecolor="k")
axes[1].set_xlabel("size sigma [arcsec]"); axes[1].set_title("Generated sizes")

axes[2].hist(gen_e[np.isfinite(gen_e)], bins=25, color="#2196F3", edgecolor="k")
axes[2].set_xlabel("|e| (unweighted moments)"); axes[2].set_title("Generated ellipticities")
fig.tight_layout()
plt.show()

## 5. The same 10 galaxies with residual PSFs interpolated at 10 positions

`PSF_3-4-F_residual.fits.gz` has the same layout as the ordinary PSF product
(a 9x9 grid of 21x21 stamps tiled into one image), so `EuclidPSFModel` reads
it unchanged and `interpolate_at(x, y)` gives the bilinear combination of the
four surrounding grid stamps, renormalised to unit sum.

We interpolate at the detector positions of 10 of the selected galaxies,
display the kernels used (residual and, for comparison, the full local PSF),
then render each generated galaxy through its residual PSF with
`render_learned_galaxy` — the exact function the scene model calls per source.

> Section 6 shows that this function's coordinate convention is itself the
> thing to question: keep an eye on the orientation of the rendered galaxies
> compared with the decoded ones from section 4.

In [ ]:
with fits.open(config.data.psf_path) as hdul:
    psf_full_model = EuclidPSFModel(hdul[QUADRANT].data.astype(np.float32))
with fits.open(learned_morphology.psf_residual_path) as hdul:
    psf_res_model = EuclidPSFModel(hdul[QUADRANT].data.astype(np.float32))

print("PSF grid:", psf_res_model.grid_ny, "x", psf_res_model.grid_nx,
      "stamps of", psf_res_model.stamp_size, "px")

psf_indices = np.asarray(gal_indices[:N_SHOW])
psf_positions = pos0[psf_indices]                     # (N_SHOW, 2) detector px
psf_residuals = np.stack([psf_res_model.interpolate_at(x, y) for x, y in psf_positions])
psf_fulls = np.stack([psf_full_model.interpolate_at(x, y) for x, y in psf_positions])

# Same interpolation the data loader already ran for these sources.
np.testing.assert_allclose(
    psf_residuals,
    np.asarray(data.psf_residual_images)[psf_indices, 0],
    rtol=1e-5, atol=1e-7,
)
print("interpolate_at matches ExposureSet.psf_residual_images OK")
print(f"peak  full PSF: {psf_fulls.max(axis=(1, 2)).mean():.3f} | "
      f"residual PSF: {psf_residuals.max(axis=(1, 2)).mean():.3f}  "
      "(the residual is sharper: the isotropic part is already in decode(z))")

In [ ]:
fig, axes = plt.subplots(2, N_SHOW, figsize=(2.0 * N_SHOW, 4.6))
for k in range(N_SHOW):
    x, y = psf_positions[k]
    axes[0, k].imshow(psf_residuals[k], origin="lower", cmap="viridis")
    axes[0, k].set_title(f"({x:.0f}, {y:.0f})", fontsize=8)
    axes[1, k].imshow(psf_fulls[k], origin="lower", cmap="viridis")
    for row in (0, 1):
        axes[row, k].set_xticks([])
        axes[row, k].set_yticks([])
axes[0, 0].set_ylabel("residual PSF", fontsize=9)
axes[1, 0].set_ylabel("full PSF", fontsize=9)
fig.suptitle(
    "Interpolated 21x21 PSF stamps at 10 galaxy positions — "
    "top: residual PSF (used by the learned tier), bottom: full local PSF",
    fontsize=12,
)
fig.tight_layout()
plt.show()

In [ ]:
gsparams = galsim.GSParams(minimum_fft_size=128, maximum_fft_size=128)

galaxies_psf = jnp.stack([
    render_learned_galaxy(
        z_gen[k],
        jnp.float32(0.0), jnp.float32(0.0),          # no shear applied here
        jnp.asarray(psf_residuals[k]),               # interpolated residual PSF
        jnp.asarray(data.wcs_jacobians[psf_indices[k], 0]),  # real local WCS Jacobian
        jnp.float32(0.0), jnp.float32(0.0),          # no sub-pixel offset
        jnp.array(True),                             # source visible
        ae, ae.nx, config.data.pixel_scale, gsparams,
    )
    for k in range(N_SHOW)
])

assert jnp.all(jnp.isfinite(galaxies_psf))

fig, axes = plt.subplots(3, N_SHOW, figsize=(2.0 * N_SHOW, 6.6))
for k in range(N_SHOW):
    before, after = np.asarray(galaxies_nopsf[k]), np.asarray(galaxies_psf[k])
    axes[0, k].imshow(before, origin="lower", cmap="inferno")
    axes[1, k].imshow(after, origin="lower", cmap="inferno")
    axes[2, k].imshow(after - before, origin="lower", cmap="RdBu_r",
                      vmin=-np.abs(after - before).max(), vmax=np.abs(after - before).max())
    pa_before = second_moments(before, config.data.pixel_scale)[2]
    pa_after = second_moments(after, config.data.pixel_scale)[2]
    axes[0, k].set_title(f"z#{k}\nPA {pa_before:+.0f} -> {pa_after:+.0f} deg", fontsize=8)
    for row in range(3):
        axes[row, k].set_xticks([])
        axes[row, k].set_yticks([])
for row, label in enumerate(["no PSF", "* residual PSF", "difference"]):
    axes[row, 0].set_ylabel(label, fontsize=9)
fig.suptitle("Generated galaxies before / after render_learned_galaxy with the residual PSF",
             fontsize=13)
fig.tight_layout()
plt.show()

print("flux ratio after/before (convolution should roughly conserve flux):")
print(np.round(np.asarray(galaxies_psf.sum(axis=(1, 2)) / galaxies_nopsf.sum(axis=(1, 2))), 3))
print("\nIf the position angles above moved by a large, near-constant amount, "
      "that is the rendering-convention problem section 6 investigates.")

## 6. Rendering conventions: which frame does `decode(z)` live in?

This section exists because the MAP fits came out looking like *rotated*
galaxies with high-frequency structure radiating from the centre. That is a
rendering symptom, not an optimisation one, so it is worth isolating before
fitting anything.

**The claim to test.** `decode(z)` was trained against `sci_subtracted`
cutouts, i.e. plain `Cutout2D` slices of the science image: a **detector
pixel grid**. Training then drew the model with
`drawImage(nx, ny, scale=0.1, method="no_pixel")` and
`GSParams(minimum_fft_size=64, maximum_fft_size=64)` — a pure pixel scale,
no rotation. `render_learned_galaxy`, on the other hand, wraps the decoded
array as `InterpolatedImage(Image(g, scale=0.1))` — which declares it a
**sky-plane** profile — and draws it through the local WCS Jacobian. If that
Jacobian carries a rotation, the model galaxy comes out rotated with respect
to the observed one.

Three differences between the two conventions, all testable below:

| | AE training (`convolve_galsim`) | `render_learned_galaxy` today |
|---|---|---|
| frame | `scale=0.1`, no rotation | `wcs=Jacobian` |
| pixel response | `method="no_pixel"` | `method="auto"` (extra pixel convolution) |
| FFT grid | 64 (= stamp size, so wrap-around) | 128 |

Note the PSF stamps raise the same question: they are detector-grid arrays
too, and the parametric tier treats them as sky-plane profiles as well.

In [ ]:
def decompose_jacobian(j):
    # (dudx, dudy, dvdx, dvdy) -> rotation angle, the two axis scales, parity
    M = np.array([[float(j[0]), float(j[1])], [float(j[2]), float(j[3])]])
    det = np.linalg.det(M)
    angle = np.degrees(np.arctan2(M[1, 0], M[0, 0]))
    sx, sy = np.hypot(M[0, 0], M[1, 0]), np.hypot(M[0, 1], M[1, 1])
    return angle, sx, sy, det


print("Local WCS Jacobian at the first fitted galaxy, per exposure:")
for j in range(data.n_exposures):
    jac = np.asarray(data.wcs_jacobians[psf_indices[0], j])
    angle, sx, sy, det = decompose_jacobian(jac)
    print(f"  exposure {j}: [[{jac[0]:+.5f} {jac[1]:+.5f}] [{jac[2]:+.5f} {jac[3]:+.5f}]] "
          f"-> rotation {angle:+.2f} deg, scales ({sx:.4f}, {sy:.4f})\"/px, "
          f"parity flip {det < 0}")

wcs_angle_deg = decompose_jacobian(np.asarray(data.wcs_jacobians[psf_indices[0], 0]))[0]
print(f"\nThis is not a diag(0.1, 0.1) matrix: it is a {wcs_angle_deg:+.1f} deg rotation.")
print("Drawing a detector-grid array through it rotates the model by that angle.")

In [ ]:
PIXEL_SCALE = config.data.pixel_scale


def _interp(arr, gsp, scale=None):
    scale = PIXEL_SCALE if scale is None else scale
    return galsim.InterpolatedImage(galsim.Image(arr, scale=scale), gsparams=gsp)


def render_library(z_i, psf_img, wcs_params, dx, dy, g1=0.0, g2=0.0, log_amp=0.0,
                   fft_size=128, method="auto"):
    # Exactly what shine.morphology.render.render_learned_galaxy does today:
    # the decoded array is treated as a sky-plane profile and drawn through
    # the WCS Jacobian.
    gsp = galsim.GSParams(minimum_fft_size=fft_size, maximum_fft_size=fft_size)
    gal = _interp(ae.decode(z_i, key=None)[0], gsp).shear(g1=g1, g2=g2)
    psf = _interp(psf_img, gsp)
    wcs = galsim.JacobianWCS(dudx=wcs_params[0], dudy=wcs_params[1],
                             dvdx=wcs_params[2], dvdy=wcs_params[3])
    stamp = galsim.Convolve([gal, psf], gsparams=gsp).drawImage(
        nx=ae.nx, ny=ae.ny, wcs=wcs,
        offset=galsim.PositionD(dx / PIXEL_SCALE, dy / PIXEL_SCALE),
        method=method,
    ).array
    return jnp.exp(log_amp) * stamp


def render_detector(z_i, psf_img, wcs_params, dx, dy, g1=0.0, g2=0.0, log_amp=0.0,
                    fft_size=64, method="no_pixel"):
    # The AE's own training convention: the decoded galaxy and the PSF stamp
    # are both detector-grid arrays, so they are drawn with a plain pixel
    # scale and never rotated. The shear is a physical, sky-frame quantity,
    # so it is rotated INTO the detector frame before being applied:
    #   e_detector = e_sky * exp(-2i*theta),  theta = WCS rotation angle.
    gsp = galsim.GSParams(minimum_fft_size=fft_size, maximum_fft_size=fft_size)
    theta = jnp.arctan2(wcs_params[2], wcs_params[0])
    c, s = jnp.cos(2 * theta), jnp.sin(2 * theta)
    g1d, g2d = g1 * c + g2 * s, -g1 * s + g2 * c
    gal = _interp(ae.decode(z_i, key=None)[0], gsp).shear(g1=g1d, g2=g2d)
    psf = _interp(psf_img, gsp)
    stamp = galsim.Convolve([gal, psf], gsparams=gsp).drawImage(
        nx=ae.nx, ny=ae.ny, scale=PIXEL_SCALE,
        offset=galsim.PositionD(dx / PIXEL_SCALE, dy / PIXEL_SCALE),
        method=method,
    ).array
    return jnp.exp(log_amp) * stamp


def render_sky(z_i, psf_img, wcs_params, dx, dy, g1=0.0, g2=0.0, log_amp=0.0,
               fft_size=128, method="no_pixel"):
    # The same physics written in the sky frame, and the version worth porting
    # to the library: map BOTH detector-grid arrays (galaxy and PSF) onto the
    # sky with the WCS Jacobian, apply the sky-frame shear there, then draw
    # back through the same WCS. Used here to verify render_detector's shear
    # rotation -- the two must agree.
    gsp = galsim.GSParams(minimum_fft_size=fft_size, maximum_fft_size=fft_size)
    j0, j1, j2, j3 = (wcs_params[0], wcs_params[1], wcs_params[2], wcs_params[3])
    gal = _interp(ae.decode(z_i, key=None)[0], gsp, scale=1.0)
    gal = gal.transform(j0, j1, j2, j3).shear(g1=g1, g2=g2)
    psf = _interp(psf_img, gsp, scale=1.0).transform(j0, j1, j2, j3)
    wcs = galsim.JacobianWCS(dudx=j0, dudy=j1, dvdx=j2, dvdy=j3)
    stamp = galsim.Convolve([gal, psf], gsparams=gsp).drawImage(
        nx=ae.nx, ny=ae.ny, wcs=wcs,
        offset=galsim.PositionD(dx / PIXEL_SCALE, dy / PIXEL_SCALE),
        method=method,
    ).array
    return jnp.exp(log_amp) * stamp


print("renderers defined: render_library (current), render_detector (training "
      "convention), render_sky (proposed library fix)")

In [ ]:
# --- Test 1: is the artefact in the decoder or in the rendering? -----------
# decode(z) alone, no GalSim at all. If the high-frequency structure is
# already here, the decoder is being driven outside its trained domain and no
# rendering change will help.
k = 0
z_test = z_gen[k]
psf_test = jnp.asarray(psf_residuals[k])
wcs_test = jnp.asarray(data.wcs_jacobians[psf_indices[k], 0])

raw = np.asarray(ae.decode(z_test, key=None)[0])
lib = np.asarray(render_library(z_test, psf_test, wcs_test, 0.0, 0.0))
det = np.asarray(render_detector(z_test, psf_test, wcs_test, 0.0, 0.0))
sky = np.asarray(render_sky(z_test, psf_test, wcs_test, 0.0, 0.0))

panels = [(raw, "decode(z) — no GalSim"),
          (lib, "render_library (WCS frame)"),
          (det, "render_detector (training frame)"),
          (sky, "render_sky (proposed fix)")]

fig, axes = plt.subplots(1, 4, figsize=(19, 4.6))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(np.arcsinh(img / max(img.max() * 1e-3, 1e-8)), origin="lower", cmap="inferno")
    sigma, e, pa = second_moments(img, PIXEL_SCALE)
    ax.set_title(f"{title}\nPA={pa:+.1f} deg, |e|={e:.3f}, sigma={sigma:.2f}\"", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Same latent code, four rendering conventions (arcsinh stretch)", fontsize=13)
fig.tight_layout()
plt.show()

pa_raw = second_moments(raw, PIXEL_SCALE)[2]
pa_lib = second_moments(lib, PIXEL_SCALE)[2]
pa_det = second_moments(det, PIXEL_SCALE)[2]
delta = (pa_lib - pa_raw + 90) % 180 - 90
print(f"position angle: decode(z) {pa_raw:+.1f} deg | render_library {pa_lib:+.1f} deg "
      f"| render_detector {pa_det:+.1f} deg")
print(f"  library rotates the decoded galaxy by {delta:+.1f} deg; "
      f"the WCS rotation is {wcs_angle_deg:+.1f} deg")

In [ ]:
# --- Test 2: does render_detector's shear rotation match the sky frame? ----
# render_sky applies the shear in sky coordinates, which is the definition we
# want. render_detector applies a rotated shear on the detector grid. With a
# non-zero test shear the two must give the same stamp; if the rotation sign
# were wrong they would visibly disagree.
g_test = (0.10, -0.06)


def _norm(a):
    a = np.asarray(a)
    return a / a.sum()


sky_sheared = _norm(render_sky(z_test, psf_test, wcs_test, 0.0, 0.0, *g_test, method="no_pixel"))
# Same FFT grid and pixel method on both sides: this test is about the shear
# rotation only, not about the conventions tested below.
det_kwargs = dict(fft_size=128, method="no_pixel")
det_sheared = _norm(render_detector(z_test, psf_test, wcs_test, 0.0, 0.0,
                                    *g_test, **det_kwargs))
det_wrongsign = _norm(render_detector(z_test, psf_test, wcs_test, 0.0, 0.0,
                                      g_test[0], -g_test[1], **det_kwargs))

for label, img in (("rotated shear (as implemented)", det_sheared),
                   ("deliberately wrong sign      ", det_wrongsign)):
    diff = np.abs(img - sky_sheared).max() / sky_sheared.max()
    print(f"{label}: max relative difference vs sky frame = {diff:.4f}")
print("\nThe first line should be far smaller than the second; that is what "
      "pins the sign of the sky->detector shear rotation.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (img, title) in zip(axes, [(sky_sheared, "sky frame, shear on sky"),
                                   (det_sheared, "detector frame, rotated shear"),
                                   (det_sheared - sky_sheared, "difference")]):
    lim = np.abs(img).max()
    ax.imshow(img, origin="lower", cmap="RdBu_r" if "diff" in title else "inferno",
              vmin=-lim if "diff" in title else None, vmax=lim)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
plt.show()

In [ ]:
# --- Test 3: the pixel-response and FFT-size knobs ------------------------
# Training convolved with method="no_pixel" on a 64-point FFT grid (the same
# size as the stamp, so the convolution wraps around); the library uses
# method="auto" (one extra pixel convolution) on a 128-point grid.
variants = {
    "training: no_pixel, FFT 64": dict(method="no_pixel", fft_size=64),
    "no_pixel, FFT 128": dict(method="no_pixel", fft_size=128),
    "auto (extra pixel), FFT 128": dict(method="auto", fft_size=128),
}
rendered = {k: np.asarray(render_detector(z_test, psf_test, wcs_test, 0.0, 0.0, **v))
            for k, v in variants.items()}
reference = rendered["training: no_pixel, FFT 64"]

fig, axes = plt.subplots(2, len(variants), figsize=(5 * len(variants), 8))
for col, (label, img) in enumerate(rendered.items()):
    axes[0, col].imshow(np.arcsinh(img / max(img.max() * 1e-3, 1e-8)),
                        origin="lower", cmap="inferno")
    axes[0, col].set_title(label, fontsize=9)
    d = img - reference
    lim = max(np.abs(d).max(), 1e-12)
    axes[1, col].imshow(d, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[1, col].set_title(f"minus training convention\nmax |diff| / peak = "
                           f"{np.abs(d).max() / img.max():.3f}", fontsize=9)
    for row in (0, 1):
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
fig.suptitle("Pixel-response and FFT-grid conventions, detector frame", fontsize=13)
fig.tight_layout()
plt.show()

# Ringing check: how much flux sits in the outer border of the stamp, where an
# isolated galaxy should have none? FFT wrap-around and edge discontinuities
# both show up here.
border = np.ones((ae.nx, ae.ny), dtype=bool)
border[6:-6, 6:-6] = False
for label, img in rendered.items():
    print(f"{label:30s}: border flux / total = {img[border].sum() / img.sum():.5f}, "
          f"min pixel = {img.min():+.3e}")

## 7. Does a free flux parameter fix the fit? Four models, same galaxies

Now the decisive experiment. The same galaxies, the same stamps, the same
optimiser — only the *model* changes. It is a 2x2: two rendering frames
crossed with "free per-galaxy flux" versus "no flux parameter at all".

| variant | renderer | free flux? | what it isolates |
|---|---|---|---|
| **A** | `render_library` | no | the library exactly as it is today |
| **B** | `render_library` | yes | does a flux parameter alone rescue it? |
| **C** | `render_detector` (training convention) | no | **can the model fit these galaxies with no flux freedom at all?** |
| **D** | `render_detector` (training convention) | yes | frame fix and flux together |

A/B and C/D are the same comparison run in two frames, which is what makes
the answer about the flux parameter frame-independent. **C is the variant the
faint selection of section 2 exists for**: with the observed galaxies now
drawn from the same population the flow was trained on, the decoder's
intrinsic amplitude is roughly the right one, so "no flux parameter" is no
longer obviously hopeless — the question is whether it is *good enough*.

Free per galaxy in every variant: the latent `z` (from the flow prior), and a
sub-pixel recentring `dx`, `dy` added on top of the known offset
`pos - round(pos)`. The shear is held at `g1 = g2 = 0` throughout — this
section is about whether the model can represent the galaxies at all, and
nothing about the morphology is allowed to leak into a shear estimate.

`log_amp` is the flux parameter: the rendered stamp is multiplied by
`exp(log_amp)`. Variants A and C have no such freedom, which is exactly the
situation in `_render_tier` today, where the learned tier ignores the sampled
`flux`.


In [ ]:
import numpyro
import numpyro.distributions as dist
from shine.morphology.prior import sample_latent_codes

PER_GALAXY_MAP_STEPS = 400
PER_GALAXY_LR = 0.01

fit_indices = np.asarray(gal_indices)
n_fit = len(fit_indices)

obs_stamps = jnp.stack([jnp.asarray(cutout(image0, *pos0[i])) for i in fit_indices])
sigma_stamps = jnp.stack([jnp.asarray(cutout(np.asarray(data.noise_sigma[0]), *pos0[i]))
                          for i in fit_indices])
psf_stamps = jnp.asarray(np.asarray(data.psf_residual_images)[fit_indices, 0])
wcs_stamps = jnp.asarray(np.asarray(data.wcs_jacobians)[fit_indices, 0])

# Known sub-pixel residual of each source w.r.t. its (rounded) stamp centre,
# in arcsec. dx/dy below are free corrections on top of this.
subpix = jnp.asarray((pos0[fit_indices] - np.round(pos0[fit_indices])) * PIXEL_SCALE)

# Initial amplitude: observed stamp flux / decoded flux at z_base = 0.
z_init = flow.unflatten_latent(jax.vmap(flow.forward)(
    base_loc + base_scale * jnp.zeros((n_fit, latent_flat))))
decoded_init = jax.vmap(lambda zi: ae.decode(zi, key=None))(z_init)[:, 0]
amp_init = jnp.clip(obs_stamps.sum(axis=(1, 2)), 1.0) / jnp.clip(decoded_init.sum(axis=(1, 2)), 1e-6)

print(f"Fitting {n_fit} galaxies independently on {obs_stamps.shape[1]}x{obs_stamps.shape[2]} stamps")
print(f"observed stamp flux [ADU]: median {float(jnp.median(obs_stamps.sum(axis=(1, 2)))):.3e}")
print(f"initial amplitude guess  : median {float(jnp.median(amp_init)):.1f}")

In [ ]:
def make_stamp_model(renderer, free_amp, render_kwargs):
    def render_all(z, log_amp, dx, dy):
        def one(z_i, p, w, la, dxi, dyi, sp):
            return renderer(z_i, p, w, sp[0] + dxi, sp[1] + dyi,
                            log_amp=la, **render_kwargs)
        return jax.vmap(one)(z, psf_stamps, wcs_stamps, log_amp, dx, dy, subpix)

    def model(observed_data=None, **extra_args):
        with numpyro.plate("galaxies", n_fit):
            z = sample_latent_codes("z", flow, n_fit)
            dx = numpyro.sample("dx", dist.Normal(0.0, 0.1))   # arcsec, ~1 px
            dy = numpyro.sample("dy", dist.Normal(0.0, 0.1))
            if free_amp:
                # Very weak: the order of magnitude is what we are measuring.
                log_amp = numpyro.sample("log_amp", dist.Normal(0.0, 10.0))
            else:
                log_amp = jnp.zeros(n_fit)

        model_stamps = render_all(z, log_amp, dx, dy)
        numpyro.sample("obs", dist.Normal(model_stamps, sigma_stamps).to_event(3),
                       obs=observed_data)

    return model, render_all


# Two frames x (no flux param / free flux param). A and C are the two
# "no flux" arms -- the question the faint selection of section 2 was made to
# answer -- and B/D their free-flux counterparts in the same frame.
VARIANTS = [
    ("A. library renderer, no flux param", render_library, False,
     dict(fft_size=128, method="auto")),
    ("B. library renderer + flux param", render_library, True,
     dict(fft_size=128, method="auto")),
    ("C. training frame, no flux param", render_detector, False,
     dict(fft_size=64, method="no_pixel")),
    ("D. training frame + flux param", render_detector, True,
     dict(fft_size=64, method="no_pixel")),
]

results = {}
for label, renderer, free_amp, kwargs in VARIANTS:
    model, render_all = make_stamp_model(renderer, free_amp, kwargs)
    init = {
        "z_base": jnp.zeros((n_fit, latent_flat)),
        "dx": jnp.zeros(n_fit),
        "dy": jnp.zeros(n_fit),
    }
    if free_amp:
        init["log_amp"] = jnp.log(amp_init)

    t0 = time.time()
    est = Inference(
        model,
        InferenceConfig(
            method="map",
            map_config=MAPConfig(enabled=True, num_steps=PER_GALAXY_MAP_STEPS,
                                 learning_rate=PER_GALAXY_LR),
            rng_seed=RNG_SEED,
        ),
    ).run_map(jax.random.PRNGKey(RNG_SEED), observed_data=obs_stamps, init_params=init)

    z_fit = flow.unflatten_latent(jax.vmap(flow.forward)(
        base_loc + base_scale * est["z_base"]))
    log_amp_fit = est["log_amp"] if free_amp else jnp.zeros(n_fit)
    stamps_fit = render_all(z_fit, log_amp_fit, est["dx"], est["dy"])
    chi = (obs_stamps - stamps_fit) / sigma_stamps
    chi2 = np.asarray(jnp.mean(chi ** 2, axis=(1, 2)))

    results[label] = dict(est=est, z=z_fit, log_amp=log_amp_fit, stamps=stamps_fit,
                          chi=chi, chi2=chi2, renderer=renderer, kwargs=kwargs,
                          free_amp=free_amp, render_all=render_all)
    print(f"{label:38s} median chi2/px = {np.median(chi2):9.2f}  "
          f"({time.time() - t0:.0f} s)")

In [ ]:
labels = list(results)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))

axes[0].boxplot([np.log10(np.clip(results[l]["chi2"], 1e-3, None)) for l in labels])
axes[0].set_xticklabels([l.split(".")[0] for l in labels])
axes[0].axhline(0.0, color="k", ls="--", label="chi2/px = 1")
axes[0].set_ylabel("log10(chi2 / pixel)")
axes[0].set_title("Reconstruction quality per variant")
axes[0].legend(fontsize=8)

for l in labels:
    if results[l]["free_amp"]:
        axes[1].hist(np.log10(np.exp(np.asarray(results[l]["log_amp"]))), bins=15,
                     alpha=0.55, label=l.split(".")[0], edgecolor="k")
axes[1].set_xlabel("log10(fitted flux scale)"); axes[1].set_ylabel("galaxies")
axes[1].set_title("The AE-units -> ADU factor, as measured by the fit")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

print("median chi2/pixel:")
for l in labels:
    print(f"  {l:38s} {np.median(results[l]['chi2']):10.2f}")

# The paired question: what does the flux parameter buy, frame by frame?
print("\nfree flux vs. none, at fixed renderer:")
for no_flux, free_flux in (("A", "B"), ("C", "D")):
    a = next(l for l in labels if l.startswith(no_flux))
    b = next(l for l in labels if l.startswith(free_flux))
    ma, mb = np.median(results[a]["chi2"]), np.median(results[b]["chi2"])
    print(f"  {a.split('.')[1].strip():34s} {ma:9.2f}  ->  "
          f"{b.split('.')[1].strip():30s} {mb:9.2f}   "
          f"(factor {ma / mb:.2f})")

best = min(labels, key=lambda l: np.median(results[l]["chi2"]))
print(f"\nbest variant: {best}")
amp_best = np.exp(np.asarray(results[best]["log_amp"])) if results[best]["free_amp"] else None
if amp_best is not None:
    spread = np.percentile(amp_best, 84) / np.percentile(amp_best, 16)
    print(f"  fitted flux scale: median {np.median(amp_best):.1f}, "
          f"16-84th spread factor {spread:.2f}")
    print("  -> a spread near 1 means one global constant would reconcile the two "
          "flux conventions; a large spread means the learned tier needs its own "
          "per-galaxy flux parameter (which is what variant B/C add).")

In [ ]:
# Best and worst fits for the winning variant, plus the decoder output alone
# (to confirm whether any residual artefact comes from the decoder or the
# rendering).
res = results[best]
order = np.argsort(res["chi2"])
show = np.concatenate([order[:N_SHOW // 2], order[-(N_SHOW - N_SHOW // 2):]])

fig, axes = plt.subplots(4, len(show), figsize=(2.0 * len(show), 8.8))
for col, k in enumerate(show):
    obs = np.asarray(obs_stamps[k])
    mod = np.asarray(res["stamps"][k])
    chi = np.asarray(res["chi"][k])
    lim = float(np.percentile(np.abs(chi), 99))
    raw_k = np.asarray(ae.decode(res["z"][k], key=None)[0])

    axes[0, col].imshow(np.arcsinh(obs), origin="lower", cmap="gray_r")
    axes[1, col].imshow(np.arcsinh(mod), origin="lower", cmap="gray_r")
    axes[2, col].imshow(chi, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    axes[3, col].imshow(np.arcsinh(raw_k / max(raw_k.max() * 1e-3, 1e-8)),
                        origin="lower", cmap="inferno")
    axes[0, col].set_title(f"#{fit_indices[k]}\nchi2/px={res['chi2'][k]:.1f}", fontsize=8)
    for row in range(4):
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
for row, label in enumerate(["observed", "MAP model", "chi", "decode(z_MAP), raw"]):
    axes[row, 0].set_ylabel(label, fontsize=9)
fig.suptitle(f"{best} — {N_SHOW // 2} best and {N_SHOW - N_SHOW // 2} worst fits", fontsize=13)
fig.tight_layout()
plt.show()

## 8. A shear fit under the repaired model

If section 7 says the model can represent the galaxies, the next question is
whether a shear can be measured with it. This fits **one global `(g1, g2)`
shared by every galaxy and every exposure**, on postage stamps rather than on
the full frame — same likelihood, much cheaper, and entirely under the
notebook's control (no dependency on `_render_tier`'s conventions).

Each (galaxy, exposure) pair contributes its own stamp with its own PSF, WCS
Jacobian and sub-pixel offset; `z`, `log_amp`, `dx`, `dy` stay per galaxy and
are shared across that galaxy's exposures, which is precisely what makes
multi-exposure fitting worthwhile.

In [ ]:
# Build (galaxy, exposure) stamp pairs for every galaxy visible with a full
# cutout in that exposure.
pair_gal, pair_obs, pair_sigma, pair_psf, pair_wcs, pair_subpix = [], [], [], [], [], []
for k, i in enumerate(fit_indices):
    for j in range(data.n_exposures):
        if not bool(data.source_visible[i, j]):
            continue
        pos = np.asarray(data.pixel_positions[i, j])
        obs = cutout(np.asarray(data.images[j]), *pos)
        if obs is None:
            continue
        pair_gal.append(k)
        pair_obs.append(obs)
        pair_sigma.append(cutout(np.asarray(data.noise_sigma[j]), *pos))
        pair_psf.append(np.asarray(data.psf_residual_images[i, j]))
        pair_wcs.append(np.asarray(data.wcs_jacobians[i, j]))
        pair_subpix.append((pos - np.round(pos)) * PIXEL_SCALE)

pair_gal = jnp.asarray(np.array(pair_gal))
pair_obs = jnp.asarray(np.array(pair_obs))
pair_sigma = jnp.asarray(np.array(pair_sigma))
pair_psf = jnp.asarray(np.array(pair_psf))
pair_wcs = jnp.asarray(np.array(pair_wcs))
pair_subpix = jnp.asarray(np.array(pair_subpix))
print(f"{len(pair_gal)} (galaxy, exposure) stamps from {n_fit} galaxies "
      f"across {data.n_exposures} exposures")

best_renderer = results[best]["renderer"]
best_kwargs = results[best]["kwargs"]
print(f"using the section-7 winner: {best}")

In [ ]:
def render_pairs(z, log_amp, dx, dy, g1, g2):
    def one(z_i, p, w, la, dxi, dyi, sp):
        return best_renderer(z_i, p, w, sp[0] + dxi, sp[1] + dyi,
                             g1=g1, g2=g2, log_amp=la, **best_kwargs)
    return jax.vmap(one)(z[pair_gal], pair_psf, pair_wcs, log_amp[pair_gal],
                         dx[pair_gal], dy[pair_gal], pair_subpix)


def shear_model(observed_data=None, **extra_args):
    g1 = numpyro.sample("g1", dist.Normal(0.0, 0.05))
    g2 = numpyro.sample("g2", dist.Normal(0.0, 0.05))
    with numpyro.plate("galaxies", n_fit):
        z = sample_latent_codes("z", flow, n_fit)
        log_amp = numpyro.sample("log_amp", dist.Normal(0.0, 10.0))
        dx = numpyro.sample("dx", dist.Normal(0.0, 0.1))
        dy = numpyro.sample("dy", dist.Normal(0.0, 0.1))

    model_stamps = render_pairs(z, log_amp, dx, dy, g1, g2)
    numpyro.sample("obs", dist.Normal(model_stamps, pair_sigma).to_event(3),
                   obs=observed_data)


shear_init = {
    "g1": jnp.float32(0.0),
    "g2": jnp.float32(0.0),
    # start from the section-7 solution: same galaxies, shear-free
    "z_base": jnp.asarray(results[best]["est"]["z_base"]),
    "log_amp": jnp.asarray(results[best]["log_amp"]),
    "dx": jnp.asarray(results[best]["est"]["dx"]),
    "dy": jnp.asarray(results[best]["est"]["dy"]),
}

t0 = time.time()
shear_est = Inference(
    shear_model,
    InferenceConfig(
        method="map",
        map_config=MAPConfig(enabled=True, num_steps=PER_GALAXY_MAP_STEPS,
                             learning_rate=PER_GALAXY_LR),
        rng_seed=RNG_SEED,
    ),
).run_map(jax.random.PRNGKey(RNG_SEED), observed_data=pair_obs, init_params=shear_init)
print(f"shear MAP done in {time.time() - t0:.0f} s")

z_shear = flow.unflatten_latent(jax.vmap(flow.forward)(
    base_loc + base_scale * shear_est["z_base"]))
stamps_shear = render_pairs(z_shear, shear_est["log_amp"], shear_est["dx"],
                            shear_est["dy"], shear_est["g1"], shear_est["g2"])
chi_shear = (pair_obs - stamps_shear) / pair_sigma
print(f"\nMAP shear: g1 = {float(shear_est['g1']):+.5f}   g2 = {float(shear_est['g2']):+.5f}")
print(f"median chi2/pixel over {len(pair_gal)} stamps: "
      f"{float(jnp.median(jnp.mean(chi_shear ** 2, axis=(1, 2)))):.2f}")
print("\nA MAP point estimate carries no error bar; treat the sign and order of "
      "magnitude as diagnostics, not as a measurement. Switch the method to "
      "'nuts' for credible intervals, and validate on injected known shear first.")

## 9. The library path as it stands today

For reference, the same data through `MultiExposureScene` +
`Inference.run_map` — the code an actual SHINE run would execute. It is
variant **A** of section 7 (WCS-frame renderer, no flux freedom for the
learned tier), applied to the full 2048x2066 frames instead of stamps, so its
residuals should carry the same signature.

In [ ]:
scene = MultiExposureScene(config, data)
model = scene.build_model()
assert scene.ae is not None and scene.flow is not None

init_params = {
    "g1": jnp.float32(0.0),
    "g2": jnp.float32(0.0),
    "flux": jnp.asarray(data.catalog_flux_adu),
    "hlr": jnp.asarray(data.catalog_hlr_arcsec),
    "e1": jnp.zeros(data.n_sources),
    "e2": jnp.zeros(data.n_sources),
    "dx": jnp.zeros(data.n_sources),
    "dy": jnp.zeros(data.n_sources),
    "z_base": jnp.zeros((data.n_sources, latent_flat)),
}

t0 = time.time()
idata = Inference(model, config.inference).run(
    jax.random.PRNGKey(RNG_SEED), observed_data=data.images, init_params=init_params)
print(f"scene MAP done in {time.time() - t0:.0f} s ({MAP_STEPS} steps, "
      f"{data.n_sources} sources, {data.n_exposures} exposures)")

posterior = idata.posterior
map_params = {name: np.squeeze(posterior[name].values) for name in posterior.data_vars}
z_base_map = jnp.asarray(map_params["z_base"]).reshape(data.n_sources, latent_flat)
z_map = flow.unflatten_latent(jax.vmap(flow.forward)(base_loc + base_scale * z_base_map))
print(f"scene MAP shear: g1 = {float(map_params['g1']):+.5f}  g2 = {float(map_params['g2']):+.5f}")
print(f"section 8 shear: g1 = {float(shear_est['g1']):+.5f}  g2 = {float(shear_est['g2']):+.5f}")

In [ ]:
render_params = dict(map_params)
render_params["z"] = z_map
model_images = render_model_images(
    render_params, data,
    pixel_scale=config.data.pixel_scale,
    stamp_sizes=config.galaxy_stamp_sizes,
    ae=scene.ae, learned_tier_idx=0,
)

# Bit 5 = GHOST, 18 = STARSIGNAL, 19 = SATURATEDSTAR: not modelled by the scene.
RESIDUAL_EXCLUDE_BITS = 0x1 | (1 << 5) | (1 << 18) | (1 << 19)
for j in range(data.n_exposures):
    fig = plot_exposure_comparison(
        observed=data.images[j], model=model_images[j],
        noise_sigma=data.noise_sigma[j], mask=data.masks[j], exposure_idx=j,
        residual_mask=(data.flag_maps[j] & RESIDUAL_EXCLUDE_BITS) == 0,
    )
    plt.show()

for j in range(data.n_exposures):
    m = np.asarray((data.flag_maps[j] & RESIDUAL_EXCLUDE_BITS) == 0)
    chi = (np.asarray(data.images[j]) - np.asarray(model_images[j])) / np.asarray(data.noise_sigma[j])
    print(f"exposure {j}: chi2/pixel = {np.nanmean(chi[m] ** 2):.3f} over {m.sum()} pixels")

## 10. What this notebook found, and what to change in the library

**Fixed already**

1. **The latent prior was not the trained one.** `flowjax` keeps a flow's base
   distribution `loc`/`scale` trainable and `train_flow.py` does not freeze
   them, so `wandb_weights/2815kuay/epoch_500` (the RealNVP shipped before the
   current flow) had a base `loc` reaching `-1.65`, not `N(0, 1)`.
   `sample_latent_codes` pushed a standard normal through `flow.forward`,
   sampling a prior whose per-dimension means were off by up to 1.64
   (percentiles up to 3.84), with 0.16% of latents outside the `(-5, 5)`
   range `softclip2` guarantees the decoder ever saw. Fixed in
   `shine/morphology/prior.py`. The current MAF checkpoint `9i28jqsm` happens
   to have stayed near-standard (max `|loc|` 0.02, scales 0.90-1.04), so the
   fix is now invisible on this checkpoint — section 4 re-checks it on
   whichever checkpoint is loaded, so a future one cannot reintroduce it
   silently.
2. **The flow is now the architecture `train_flow.py` describes.**
   `9i28jqsm/epoch_500` is MAF + rational-quadratic splines on `interval: 5.0`,
   replacing the RealNVP-with-affine-transformer `2815kuay`. The reason that
   mattered: `softclip2` bounds the latents to `(-5, 5)` and piles mass at the
   edges, which an affine flow cannot represent — it put 0.01% of its own
   draws outside that interval, where the decoder has never been trained.
   Measured on 2000 draws from `9i28jqsm`: **0.000%** outside. Diversity is
   comparable or better (covariance eigenvalues 5.11, 2.10, 1.84, ...; top-3 =
   56% of the variance).
   *Loading it needs `flowjax >= 18`* — 18.0.0 changed
   `RationalQuadraticSpline`'s parameterisation from 38 to 40 parameters per
   latent dimension at `knots=12`, and the checkpoint will not deserialise
   under 17.x. Note also that the flow's `config.yaml` records
   `flow_layers: 4` but the checkpoint has 8 stacked conditioners: neither
   `train_flow.py` nor `LatentFlow` forwards `flow_layers` on the MAF branch,
   so flowjax's default of 8 is what both sides build. See the comment in
   `shine/morphology/nn/flow.py`.
3. **The galaxies fitted here were the wrong ones.** `_select_sources` used to
   sort by SNR descending and truncate, so this notebook fitted the
   `MAX_SOURCES` *brightest* sources of the quadrant — median catalogue flux
   7.4e4 ADU — against a flow prior that generates ~1.3e3 ADU. That is a
   factor ~60 outside the decoder's trained domain, and it accounts for most
   of the "400-1200x flux gap" recorded in `data/LEARNED_MORPHOLOGY_NOTES.md`:
   the *core* of the same catalogue sits at median 1.1e3 ADU, on top of the
   prior. Section 2 now selects an SNR band (12-25) drawn at random, via the
   new `SourceSelectionConfig.max_snr` / `selection_order` / `selection_seed`
   fields. **The AE's output units are ADU** to within the population scatter.

**Diagnosed here, not yet fixed in the library**

4. **The learned tier renders in the wrong frame.** `decode(z)` is a
   detector-grid image (the AE trained on `Cutout2D` slices, drawn with
   `scale=0.1`, no rotation), but `render_learned_galaxy` declares it a
   sky-plane profile and draws it through the WCS Jacobian — a **58 deg
   rotation** on this quadrant (section 6, test 1). The parametric tier is
   unaffected: an `Exponential` really is defined on the sky. The fix is
   `render_sky` in section 6: map the decoded array *and* the PSF stamp onto
   the sky with the Jacobian, shear there, then draw back — equivalently,
   `render_detector`, which stays on the detector grid and rotates the shear
   instead (section 6, test 2 verifies the two agree).
5. **The PSF stamps raise the same question**, for both tiers: they are
   detector-grid arrays treated as sky-plane profiles. Their ellipticity is
   only ~0.02, but that is the same order as the shear signal, so it is worth
   settling deliberately rather than by default.
6. **The learned tier has no flux freedom.** `_render_tier` samples `flux` and
   ignores it on that tier, so the stamp amplitude is whatever `decode(z)`
   happens to produce. Section 7 is the 2x2 that measures what that costs:
   variants A and C have no flux parameter (A is the library today), B and D
   add nothing but `exp(log_amp)` in the same frame. Since amplitude and shear
   are nearly orthogonal, an explicit per-galaxy flux makes any residual
   AE-units/ADU calibration irrelevant — and it stops the optimiser from
   distorting `z` (hence the morphology, hence the shear) to fabricate
   brightness. **With the faint selection of section 2 the "no flux
   parameter" arms are no longer obviously hopeless**, which is exactly what
   C is there to test; running section 7 is what settles it.
7. **The pixel-response and FFT conventions differ from training**
   (`method="auto"` vs `no_pixel`, FFT 128 vs 64). Section 6 test 3 shows how
   large that is; the A/B (library) versus C/D (training frame) split in
   section 7 shows whether it matters for the fit.

**Suggested order of work**

Run section 7 on a GPU runtime and read off the two paired comparisons (A vs
B, C vs D). Then port the frame fix (`render_sky`) and, if the paired
comparison says it earns its place, the flux parameter into
`shine/morphology/render.py` + `shine/euclid/scene.py`; rerun section 7 to
confirm the library path now matches the winning variant, then validate the
whole chain on injected known shear before trusting any `g1/g2` from real
data.
